In [ ]:
import IPython.display
import numpy as np
import astropy.units as u
import astropy.visualization
import matplotlib.pyplot as plt
import named_arrays as na
import msfc_ccd

In [ ]:
astropy.visualization.quantity_support();

In [ ]:
dark = msfc_ccd.fits.open(msfc_ccd.samples.path_dark_esis1)

fig, ax = plt.subplots(
    figsize=(8, 4),
    constrained_layout=True,
)
im = na.plt.imshow(
    dark.outputs.value,
    axis_x=dark.axis_x,
    axis_y=dark.axis_y,
    ax=ax,
)
ax.set_xlabel("detector $x$ (pix)")
ax.set_ylabel("detector $y$ (pix)")
plt.colorbar(im.ndarray.item(), ax=ax, label="signal (DN)");

In [ ]:
taps = dark.taps

In [ ]:
axis_tap_x = taps.axis_tap_x
axis_tap_y = taps.axis_tap_y

In [ ]:
fig, ax = na.plt.subplots(
    axis_rows=axis_tap_y,
    axis_cols=axis_tap_x,
    nrows=taps.shape[axis_tap_y],
    ncols=taps.shape[axis_tap_x],
    sharex=True,
    constrained_layout=True,
)
na.plt.plot(
    taps.outputs.mean_trimmed(.01, taps.axis_y),
    axis=taps.axis_x,
    ax=ax,
)
na.plt.set_ylim(
    bottom=taps.outputs.percentile(5, axis=taps.axis_xy),
    top=taps.outputs.percentile(95, axis=taps.axis_xy),
    ax=ax,
)
na.plt.axvspan(
    xmin=0,
    xmax=taps.camera.sensor.num_blank,
    color="green",
    alpha=0.2,
    ax=ax,
    label="blank columns",
)
na.plt.axvspan(
    xmin=taps.num_x - taps.camera.sensor.num_overscan,
    xmax=taps.num_x,
    color="red",
    alpha=0.2,
    ax=ax,
    label="overscan columns",
)
na.plt.set_ylabel("row-averaged signal (DN)", ax[{axis_tap_x: 0}])
na.plt.set_xlabel("columns", ax=ax[{axis_tap_y: 0}])
na.plt.text(
    x=0.9,
    y=0.95,
    s=taps.label,
    ax=ax,
    transform=na.plt.transAxes(ax),
    ha="right",
    va="top",
)
ax.ndarray.flat[0].legend();

In [ ]:
bias_blank = taps.bias(num_blank=None, num_overscan=0)
bias_overscan = taps.bias(num_blank=0, num_overscan=None)

In [ ]:
unbiased_blank = taps - bias_blank
unbiased_overscan = taps - bias_overscan

In [ ]:
kwargs_filter = dict(
    size={taps.axis_x: 11, taps.axis_y: 11},
    proportion=0.05,
)
unbiased_blank = na.ndfilters.trimmed_mean_filter(unbiased_blank, **kwargs_filter)
unbiased_overscan = na.ndfilters.trimmed_mean_filter(unbiased_overscan, **kwargs_filter)

In [ ]:
dark_blank = dark.from_taps(unbiased_blank)
dark_overscan = dark.from_taps(unbiased_overscan)

In [ ]:
kwargs_hist = dict(
    axis=taps.axis_xy,
    bins=na.arange(-2, 2, "xy", .1) * u.DN,
    density=True,
)
hist_blank = na.histogram(unbiased_blank.outputs, **kwargs_hist)
hist_overscan = na.histogram(unbiased_overscan.outputs, **kwargs_hist)

fig, ax = na.plt.subplots(
    axis_rows=axis_tap_y,
    axis_cols=axis_tap_x,
    nrows=taps.shape[axis_tap_y],
    ncols=taps.shape[axis_tap_x],
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
na.plt.stairs(
    hist_blank.inputs,
    hist_blank.outputs,
    ax=ax,
    axis="xy",
    label="blank",
)
na.plt.stairs(
    hist_overscan.inputs,
    hist_overscan.outputs,
    ax=ax,
    axis="xy",
    label="overscan",
)
na.plt.text(
    x=0.95,
    y=0.95,
    s=taps.label,
    ax=ax,
    transform=na.plt.transAxes(ax),
    ha="right",
    va="top",
)
na.plt.axvline(0, ax=ax, color="black", linestyle="--")
na.plt.set_ylabel("probability density", ax[{axis_tap_x: 0}])
na.plt.set_xlabel("smoothed signal (DN)", ax=ax[{axis_tap_y: 0}])
ax[{axis_tap_x: 0, axis_tap_y: ~0}].ndarray.legend(loc="upper left");

In [ ]:
unbiased_blank.outputs.mean_trimmed(0.01, taps.axis_xy)

In [ ]:
unbiased_overscan.outputs.mean_trimmed(0.01, taps.axis_xy)

In [ ]:
fig, ax = plt.subplots(constrained_layout=True)
colorizer = plt.Colorizer(
    norm=plt.Normalize(
        vmin=-1,
        vmax=1,
    ),
)
ani = na.plt.pcolormovie(
    na.ScalarArray(
        ndarray=np.array(["blank", "overscan"]),
        axes="blink",
    ),
    dark.inputs.pixel.x,
    dark.inputs.pixel.y,
    C=na.stack(
        arrays=[dark_blank.outputs, dark_overscan.outputs],
        axis="blink",
    ),
    axis_time="blink",
    ax=ax,
    kwargs_pcolormesh=dict(
        colorizer=colorizer,
    ),
    kwargs_animation=dict(
        interval=1000,
    )
)

ax.set_xlabel("detector $x$ (pix)")
ax.set_ylabel("detector $y$ (pix)")
plt.colorbar(
    mappable=plt.cm.ScalarMappable(colorizer=colorizer), 
    ax=ax,
    label="signal (DN)"
)
plt.close(fig)
IPython.display.HTML(ani.to_jshtml())

In [ ]:
led = msfc_ccd.fits.open(msfc_ccd.samples.path_led_esis1)
led_dark = msfc_ccd.fits.open(msfc_ccd.samples.path_led_dark_esis1)

fig, ax = plt.subplots(
    figsize=(8, 4),
    constrained_layout=True,
)
im = na.plt.imshow(
    led.outputs.value,
    axis_x=led.axis_x,
    axis_y=led.axis_y,
    ax=ax,
)
ax.set_xlabel("detector $x$ (pix)")
ax.set_ylabel("detector $y$ (pix)")
plt.colorbar(im.ndarray.item(), ax=ax, label="signal (DN)");

In [ ]:
taps_led = led.taps
taps_led_dark = led_dark.taps

axis_x = taps_led.axis_x
axis_y = taps_led.axis_y

signal = taps_led.outputs - taps_led_dark.outputs
signal = signal - signal.mean(
    axis=taps_led.axis_xy,
    where=taps_led.where_blank(25),
)

In [ ]:
num = 6
num_x = taps_led.num_x
num_overscan = taps_led.camera.sensor.num_overscan

fig, ax = na.plt.subplots(
    axis_rows=axis_tap_y,
    axis_cols=axis_tap_x,
    nrows=taps_led.shape[axis_tap_y],
    ncols=taps_led.shape[axis_tap_x],
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
na.plt.stairs(
    na.arange(num_x - num, num_x + 1, axis=axis_x) - 0.5,
    signal[{axis_x: slice(-num, None)}].mean(axis_y),
    axis=axis_x,
    ax=ax,
)
na.plt.axvspan(
    xmin=num_x - num_overscan - 0.5,
    xmax=num_x - 0.5,
    color="red",
    alpha=0.2,
    ax=ax,
    label="overscan columns",
)
na.plt.set_yscale("log", ax=ax)
na.plt.set_ylabel("row-averaged signal (DN)", ax[{axis_tap_x: 0}])
na.plt.set_xlabel("columns", ax=ax[{axis_tap_y: 0}])
na.plt.text(
    x=0.05,
    y=0.05,
    s=taps_led.label,
    ax=ax,
    transform=na.plt.transAxes(ax),
    ha="left",
    va="bottom",
)
ax.ndarray.flat[0].legend();

In [ ]:
edge = signal[{axis_x: ~num_overscan}]
overscan_1 = signal[{axis_x: ~1}]
overscan_2 = signal[{axis_x: ~0}]

ratio_1 = overscan_1.mean(axis_y) / edge.mean(axis_y)
ratio_1

In [ ]:
ratio_2 = overscan_2.mean(axis_y) / edge.mean(axis_y)
ratio_2

In [ ]:
kwargs_filter_y = dict(
    size={axis_y: 51},
    proportion=0.05,
)
rows = na.arange(0, taps_led.num_y, axis=axis_y)

fig, ax = na.plt.subplots(
    axis_rows=axis_tap_y,
    axis_cols=axis_tap_x,
    nrows=taps_led.shape[axis_tap_y],
    ncols=taps_led.shape[axis_tap_x],
    sharex=True,
    constrained_layout=True,
)
na.plt.plot(
    rows,
    na.ndfilters.trimmed_mean_filter(overscan_1, **kwargs_filter_y),
    axis=axis_y,
    ax=ax,
    label="first overscan column",
)
na.plt.plot(
    rows,
    na.ndfilters.trimmed_mean_filter(overscan_2, **kwargs_filter_y),
    axis=axis_y,
    ax=ax,
    label="second overscan column",
)
na.plt.plot(
    rows,
    ratio_1 * na.ndfilters.trimmed_mean_filter(edge, **kwargs_filter_y),
    axis=axis_y,
    ax=ax,
    color="black",
    linestyle="--",
    label="scaled last active column",
)
na.plt.set_ylabel("smoothed signal (DN)", ax[{axis_tap_x: 0}])
na.plt.set_xlabel("rows", ax=ax[{axis_tap_y: 0}])
na.plt.text(
    x=0.95,
    y=0.5,
    s=taps_led.label,
    ax=ax,
    transform=na.plt.transAxes(ax),
    ha="right",
    va="center",
)
handles, labels = ax.ndarray.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncols=3);

In [ ]:
kwargs_overscan = dict(num_blank=0, num_overscan=None)
taps_led.bias(**kwargs_overscan).outputs - taps_led_dark.bias(**kwargs_overscan).outputs

In [ ]:
kwargs_blank = dict(num_blank=25, num_overscan=0)
taps_led.bias(**kwargs_blank).outputs - taps_led_dark.bias(**kwargs_blank).outputs